# ShopStream Big Data Pipeline

Notebook ejecutable para demo local del pipeline de ShopStream. Replica el flujo de `pipeline.py`: limpieza, deduplicacion, imputacion, seis metricas analiticas y deteccion de anomalias. Si no hay datos locales, genera una muestra de 100 filas por tipo de evento para ejecutarse sin EMR ni S3.

In [ ]:
# Instalacion opcional para entorno local
# Ejecutar esta celda solo si faltan dependencias en el kernel.
%pip install pyspark pandas numpy faker pyarrow -q

import os
os.environ.setdefault("PYSPARK_PYTHON", "python")
print("Dependencias listas para demo local")

In [ ]:
from pathlib import Path
from uuid import uuid4
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("ShopStream-Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [ ]:
date = "2025-06-01"
input_path = "sample_data/year=2025/month=06/day=01"
output_path = "sample_output"
base_path = Path(input_path)
base_path.mkdir(parents=True, exist_ok=True)
print(f"input_path={input_path}")
print(f"output_path={output_path}")
# Setup de muestra: genera 100 filas por tipo si no existen CSV locales.
if not list(base_path.glob("*.csv")):
    rng = np.random.default_rng(42)
    users = [str(uuid4()) for _ in range(40)]
    sessions = [str(uuid4()) for _ in range(60)]
    page_types = ["home", "category", "product", "cart", "checkout"]
    countries = ["Colombia", "Mexico", "Argentina", "Chile"]
    devices = ["mobile", "desktop", "tablet"]
    categories = [f"category_{i:02d}" for i in range(1, 11)]
    products = [f"PROD-{i:04d}" for i in range(1, 51)]
    start = datetime(2025, 6, 1)

    def common(n=100):
        return pd.DataFrame({
            "user_id": rng.choice(users, n),
            "session_id": rng.choice(sessions, n),
            "timestamp": [(start + timedelta(seconds=int(x))).isoformat() for x in rng.integers(0, 86400, n)],
        })

    page_view = common()
    page_view["page_type"] = rng.choice(page_types, 100)
    page_view["page_url"] = page_view["page_type"].map(lambda p: "/" if p == "home" else f"/{p}")
    page_view["time_on_page_seconds"] = np.clip(rng.lognormal(3.6, 0.55, 100).round(), 1, 600)
    page_view["referrer"] = rng.choice(["direct", "google", "email"], 100)
    page_view["device_type"] = rng.choice(devices, 100)
    page_view["country"] = rng.choice(countries, 100)
    page_view.to_csv(base_path / "page_view.csv", index=False)

    click = common(); click["element_id"] = [f"button_{i}" for i in range(100)]; click["element_type"] = "button"
    click["page_url"] = rng.choice(["/", "/category", "/product"], 100); click["x_position"] = rng.integers(0, 1920, 100); click["y_position"] = rng.integers(0, 1080, 100)
    click.to_csv(base_path / "click.csv", index=False)

    search = common(); search["query"] = rng.choice(["laptop", "phone", "shoes"], 100); search["results_count"] = rng.integers(0, 200, 100)
    search.to_csv(base_path / "search.csv", index=False)

    product_view = common(); product_view["product_id"] = rng.choice(products, 100); product_view["category"] = rng.choice(categories, 100)
    product_view["price"] = rng.uniform(10, 1000, 100).round(2); product_view["time_on_page_seconds"] = np.clip(rng.lognormal(3.7, 0.6, 100).round(), 1, 600)
    product_view.to_csv(base_path / "product_view.csv", index=False)

    cart_event = common(); cart_event["product_id"] = rng.choice(products, 100); cart_event["action"] = rng.choice(["add", "remove"], 100, p=[0.8, 0.2])
    cart_event.to_csv(base_path / "cart_event.csv", index=False)

print("Archivos de muestra:", [p.name for p in sorted(base_path.glob("*.csv"))])

In [ ]:
# LIMPIEZA 1: leer CSVs e inferir event_type desde el nombre del archivo.
event_types = ["page_view", "click", "search", "product_view", "cart_event"]
frames = []
for event_type in event_types:
    frame = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{input_path}/{event_type}.csv")
    frames.append(frame.withColumn("event_type", F.lit(event_type)))

df = frames[0]
for frame in frames[1:]:
    df = df.unionByName(frame, allowMissingColumns=True)

df.show(5, truncate=False)

In [ ]:
# LIMPIEZA 2: deduplicar y castear timestamps a UTC.
df = df.dropDuplicates(["user_id", "session_id", "timestamp", "event_type"])
df = df.withColumn("timestamp", F.to_utc_timestamp(F.to_timestamp("timestamp"), "UTC"))
df.show(5, truncate=False)

In [ ]:
# LIMPIEZA 3: imputar nulos con mediana por page_type y category.
page_medians = df.filter(F.col("event_type") == "page_view").groupBy("page_type").agg(
    F.expr("percentile_approx(time_on_page_seconds, 0.5)").alias("median_time_on_page")
)
df = df.join(page_medians, on="page_type", how="left").withColumn(
    "time_on_page_seconds",
    F.coalesce(F.col("time_on_page_seconds").cast("double"), F.col("median_time_on_page")),
).drop("median_time_on_page")

price_medians = df.filter(F.col("event_type") == "product_view").groupBy("category").agg(
    F.expr("percentile_approx(price, 0.5)").alias("median_price")
)
df = df.join(price_medians, on="category", how="left").withColumn(
    "price", F.coalesce(F.col("price").cast("double"), F.col("median_price"))
).drop("median_price")
df.cache()
df.show(5, truncate=False)

In [ ]:
# METRICA 1: top_pages
top_pages = df.filter((F.col("event_type") == "page_view") & F.col("page_url").isNotNull()).groupBy("page_url").agg(
    F.avg("time_on_page_seconds").alias("avg_time_on_page_seconds"),
    F.count("*").alias("event_count"),
).orderBy(F.desc("avg_time_on_page_seconds")).limit(20)
top_pages.show(10, truncate=False)

In [ ]:
# METRICA 2: bounce_rate
page_views = df.filter(F.col("event_type") == "page_view")
session_counts = page_views.groupBy("session_id").agg(
    F.count("*").alias("page_view_count"),
    F.first("page_type", ignorenulls=True).alias("page_type"),
).withColumn("is_bounced", F.when(F.col("page_view_count") == 1, 1).otherwise(0))
bounce_rate = session_counts.groupBy("page_type").agg(
    F.sum("is_bounced").alias("bounced_sessions"),
    F.countDistinct("session_id").alias("total_sessions"),
).withColumn("bounce_rate", F.col("bounced_sessions") / F.col("total_sessions") * 100)
bounce_rate.show(10, truncate=False)

In [ ]:
# METRICA 3: conversion_funnel
stage1 = df.filter(F.col("event_type") == "page_view").select("user_id").distinct().count()
stage2 = df.filter(F.col("event_type") == "product_view").select("user_id").distinct().count()
stage3 = df.filter((F.col("event_type") == "cart_event") & (F.col("action") == "add")).select("user_id").distinct().count()
stage4 = df.filter((F.col("event_type") == "page_view") & (F.col("page_type") == "checkout")).select("user_id").distinct().count()
conversion_funnel = spark.createDataFrame(
    [(stage1, stage2, stage3, stage4, stage2 / stage1 * 100 if stage1 else None, stage3 / stage2 * 100 if stage2 else None, stage4 / stage3 * 100 if stage3 else None)],
    ["stage1_page_view_users", "stage2_product_view_users", "stage3_cart_add_users", "stage4_checkout_users", "stage2_over_stage1_pct", "stage3_over_stage2_pct", "stage4_over_stage3_pct"],
)
conversion_funnel.show(10, truncate=False)

In [ ]:
# METRICA 4: high_view_low_cart
views = df.filter(F.col("event_type") == "product_view").groupBy("product_id").agg(
    F.first("category", ignorenulls=True).alias("category"),
    F.avg("price").alias("avg_price"),
    F.count("*").alias("view_count"),
)
carts = df.filter((F.col("event_type") == "cart_event") & (F.col("action") == "add")).groupBy("product_id").agg(F.count("*").alias("cart_add_count"))
product_metrics = views.join(carts, "product_id", "left").fillna({"cart_add_count": 0}).withColumn("conversion_rate", F.col("cart_add_count") / F.col("view_count"))
quantiles = product_metrics.approxQuantile(["view_count", "cart_add_count"], [0.25, 0.75], 0.01)
views_p75 = quantiles[0][1] if quantiles and quantiles[0] else 0
cart_p25 = quantiles[1][0] if len(quantiles) > 1 and quantiles[1] else 0
high_view_low_cart = product_metrics.filter((F.col("view_count") > views_p75) & (F.col("cart_add_count") < cart_p25))
high_view_low_cart.show(10, truncate=False)

In [ ]:
# METRICA 5: navigation_paths
ordered = page_views.withColumn("page_event", F.struct(F.col("timestamp").alias("event_ts"), F.col("page_type").alias("page_type")))
navigation_paths = ordered.groupBy("session_id").agg(F.sort_array(F.collect_list("page_event")).alias("page_events")).withColumn(
    "path", F.concat_ws("→", F.transform("page_events", lambda item: item["page_type"]))
).groupBy("path").agg(F.count("*").alias("session_count")).orderBy(F.desc("session_count")).limit(10)
navigation_paths.show(10, truncate=False)

In [ ]:
# METRICA 6: device_country_time
device_country_time = page_views.groupBy("device_type", "country").agg(
    F.avg("time_on_page_seconds").alias("avg_time_on_page_seconds"),
    F.countDistinct("session_id").alias("distinct_sessions"),
).orderBy("device_type", "country")
device_country_time.show(10, truncate=False)

In [ ]:
# ANOMALIAS: z-score e IQR por page_type
stats_window = Window.partitionBy("page_type")
with_stats = page_views.withColumn("mean_time", F.avg("time_on_page_seconds").over(stats_window)).withColumn(
    "std_time", F.stddev("time_on_page_seconds").over(stats_window)
).withColumn("z_score", F.when(F.col("std_time") > 0, (F.col("time_on_page_seconds") - F.col("mean_time")) / F.col("std_time")).otherwise(0.0))

iqr_stats = page_views.groupBy("page_type").agg(
    F.expr("percentile_approx(time_on_page_seconds, 0.25)").alias("q1"),
    F.expr("percentile_approx(time_on_page_seconds, 0.75)").alias("q3"),
).withColumn("iqr", F.col("q3") - F.col("q1")).withColumn("lower_bound", F.col("q1") - 1.5 * F.col("iqr")).withColumn("upper_bound", F.col("q3") + 1.5 * F.col("iqr"))

anomalies = with_stats.join(iqr_stats, "page_type", "left").withColumn(
    "is_iqr_outlier", (F.col("time_on_page_seconds") < F.col("lower_bound")) | (F.col("time_on_page_seconds") > F.col("upper_bound"))
).withColumn(
    "anomaly_type",
    F.when((F.abs(F.col("z_score")) > 2) & F.col("is_iqr_outlier"), "z_score_and_iqr").when(F.abs(F.col("z_score")) > 2, "z_score").when(F.col("is_iqr_outlier"), "iqr"),
).filter((F.abs(F.col("z_score")) > 2) | (F.col("is_iqr_outlier") == True)).select("session_id", "user_id", "page_type", "time_on_page_seconds", "z_score", "is_iqr_outlier", "anomaly_type")
anomalies.show(10, truncate=False)

## Resumen final

| Output | Registros demo |
| --- | ---: |
| top_pages | ejecutar `top_pages.count()` |
| bounce_rate | ejecutar `bounce_rate.count()` |
| conversion_funnel | ejecutar `conversion_funnel.count()` |
| high_view_low_cart | ejecutar `high_view_low_cart.count()` |
| navigation_paths | ejecutar `navigation_paths.count()` |
| device_country_time | ejecutar `device_country_time.count()` |
| anomalies | ejecutar `anomalies.count()` |

Este notebook usa datos locales de muestra para defensa y demo. En EMR, el script `pipeline.py` procesa los CSV reales desde S3 y escribe Parquet snappy en el bucket processed.